# Task 173: torch_radon_ct — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CT reconstruction using torch-radon with SIRT

**Paper**: TorchRadon: Fast Differentiable Routines for Computed Tomography
**Repository**: https://github.com/matteo-ronchetti/torch-radon

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 30.01 dB
- **SSIM**: 0.8513
- **Evaluation**: 2D CT reconstruction (PSNR, SSIM, RMSE)

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
torch_radon_ct: CT Reconstruction via Radon Transform Inversion

Inverse Problem: Given a CT sinogram (Radon transform of an object),
reconstruct the original 2D image.

Forward model:  y = A*x + noise   (A = Radon transform)
Inverse:        x_hat = A^{-1}(y) (via FBP or iterative methods)

Methods implemented:
  1. Filtered Back Projection (FBP) — baseline analytical method
  2. SIRT (Simultaneous Iterative Reconstruction Technique) — iterative algebraic
  3. SIRT initialized from FBP — iterative refinement of FBP

Reference repo: https://github.com/matteo-ronchetti/torch-radon
Uses scikit-image radon/iradon as forward/adjoint operators.
"""

import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`add_gaussian_noise`, `get_circle_mask`, `main`

In [ ]:
def add_gaussian_noise(signal, snr_db):
    """Add Gaussian noise to achieve a target SNR in dB."""
    signal_power = np.mean(signal ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), signal.shape)
    return signal + noise

def get_circle_mask(size):
    """Create a circular mask for the reconstruction domain."""
    center = size // 2
    Y, X = np.ogrid[:size, :size]
    return ((X - center)**2 + (Y - center)**2) <= center**2

def main():
    print("=" * 60)
    print("CT Reconstruction via Radon Transform Inversion")
    print("=" * 60)
    np.random.seed(42)

    # ------------------------------------------------------------------
    # 1. Generate phantom (ground truth)
    # ------------------------------------------------------------------
    print("\n[1] Generating Shepp-Logan phantom...")
    phantom_full = shepp_logan_phantom()
    phantom = resize(phantom_full, (IMAGE_SIZE, IMAGE_SIZE), anti_aliasing=True)
    phantom = phantom.astype(np.float64)
    print(f"    Phantom shape: {phantom.shape}, range: [{phantom.min():.4f}, {phantom.max():.4f}]")

    # ------------------------------------------------------------------
    # 2. Forward model: Radon transform + noise
    # ------------------------------------------------------------------
    print("\n[2] Computing sinogram (Radon transform)...")
    theta = np.linspace(0, 179, NUM_ANGLES, endpoint=True)
    sinogram_clean = radon(phantom, theta=theta, circle=True)
    print(f"    Sinogram shape: {sinogram_clean.shape}")

    sinogram_noisy = add_gaussian_noise(sinogram_clean, SNR_DB)
    actual_snr = 10 * np.log10(np.mean(sinogram_clean ** 2) /
                                np.mean((sinogram_noisy - sinogram_clean) ** 2))
    print(f"    Added noise: target SNR={SNR_DB:.1f} dB, actual SNR={actual_snr:.1f} dB")

    # ------------------------------------------------------------------
    # 3. Reconstruction
    # ------------------------------------------------------------------
    results = {}
    recons = {}

    # --- FBP (Ram-Lak) ---
    print("\n[3a] FBP reconstruction (Ram-Lak)...")
    t0 = time.time()
    recon = reconstruct_fbp(sinogram_noisy, theta, IMAGE_SIZE, 'ramp')
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['FBP'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['FBP'] = recon

    # --- FBP (Shepp-Logan filter) ---
    print("\n[3b] FBP reconstruction (Shepp-Logan filter)...")
    t0 = time.time()
    recon = reconstruct_fbp(sinogram_noisy, theta, IMAGE_SIZE, 'shepp-logan')
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['FBP-SL'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['FBP-SL'] = recon

    # --- FBP (Cosine filter) ---
    print("\n[3c] FBP reconstruction (Cosine filter)...")
    t0 = time.time()
    recon = reconstruct_fbp(sinogram_noisy, theta, IMAGE_SIZE, 'cosine')
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['FBP-Cosine'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['FBP-Cosine'] = recon

    # --- FBP (Hamming filter) ---
    print("\n[3d] FBP reconstruction (Hamming filter)...")
    t0 = time.time()
    recon = reconstruct_fbp(sinogram_noisy, theta, IMAGE_SIZE, 'hamming')
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['FBP-Hamming'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['FBP-Hamming'] = recon

    # --- SIRT from zeros ---
    print(f"\n[3e] SIRT reconstruction (200 iterations, from zeros)...")
    t0 = time.time()
    recon = reconstruct_sirt(sinogram_noisy, theta, IMAGE_SIZE, n_iter=200, init_fbp=False)
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['SIRT'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['SIRT'] = recon

    # --- SIRT from FBP ---
    print(f"\n[3f] SIRT-FBP reconstruction (50 iterations, from FBP)...")
    t0 = time.time()
    recon = reconstruct_sirt(sinogram_noisy, theta, IMAGE_SIZE, n_iter=50, init_fbp=True)
    dt = time.time() - t0
    p, s, r = compute_metrics(phantom, recon)
    print(f"    PSNR={p:.2f} dB, SSIM={s:.4f}, RMSE={r:.4f}, time={dt:.2f}s")
    results['SIRT-FBP'] = {'psnr': p, 'ssim': s, 'rmse': r, 'time': dt}
    recons['SIRT-FBP'] = recon

    # ------------------------------------------------------------------
    # 4. Select best method
    # ------------------------------------------------------------------
    best_method = max(results, key=lambda k: results[k]['psnr'])
    best_recon = recons[best_method]
    best_metrics = results[best_method]

    # Best iterative
    iter_keys = [k for k in results if 'SIRT' in k]
    best_iter = max(iter_keys, key=lambda k: results[k]['psnr'])
    best_iter_recon = recons[best_iter]
    best_iter_metrics = results[best_iter]

    # Best FBP
    fbp_keys = [k for k in results if k.startswith('FBP')]
    best_fbp = max(fbp_keys, key=lambda k: results[k]['psnr'])
    best_fbp_recon = recons[best_fbp]
    best_fbp_metrics = results[best_fbp]

    print(f"\n[4] Best overall: {best_method} (PSNR={best_metrics['psnr']:.2f} dB)")
    print(f"    Best FBP: {best_fbp} (PSNR={best_fbp_metrics['psnr']:.2f} dB)")
    print(f"    Best iterative: {best_iter} (PSNR={best_iter_metrics['psnr']:.2f} dB)")

    # ------------------------------------------------------------------
    # 5. Visualization: 4-panel figure
    # ------------------------------------------------------------------
    print("\n[5] Creating visualization...")
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    # (a) Ground truth
    ax = axes[0, 0]
    im = ax.imshow(phantom, cmap='gray', vmin=0, vmax=phantom.max())
    ax.set_title(f'(a) Ground Truth\n(Shepp-Logan Phantom {IMAGE_SIZE}×{IMAGE_SIZE})',
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # (b) Sinogram
    ax = axes[0, 1]
    im = ax.imshow(sinogram_noisy, cmap='hot', aspect='auto',
                   extent=[theta[0], theta[-1], sinogram_noisy.shape[0], 0])
    ax.set_title(f'(b) Noisy Sinogram\nSNR={actual_snr:.1f} dB, {NUM_ANGLES} angles',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Angle (degrees)')
    ax.set_ylabel('Detector position')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # (c) Best FBP reconstruction
    ax = axes[1, 0]
    disp = np.clip(best_fbp_recon, 0, phantom.max())
    im = ax.imshow(disp, cmap='gray', vmin=0, vmax=phantom.max())
    ax.set_title(f'(c) {best_fbp} Reconstruction\nPSNR={best_fbp_metrics["psnr"]:.2f} dB, '
                 f'SSIM={best_fbp_metrics["ssim"]:.4f}',
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # (d) Best iterative reconstruction
    ax = axes[1, 1]
    disp = np.clip(best_iter_recon, 0, phantom.max())
    im = ax.imshow(disp, cmap='gray', vmin=0, vmax=phantom.max())
    ax.set_title(f'(d) {best_iter} Reconstruction\nPSNR={best_iter_metrics["psnr"]:.2f} dB, '
                 f'SSIM={best_iter_metrics["ssim"]:.4f}',
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle('CT Reconstruction via Radon Transform Inversion\n'
                 f'Best: {best_method} — PSNR={best_metrics["psnr"]:.2f} dB, '
                 f'SSIM={best_metrics["ssim"]:.4f}',
                 fontsize=15, fontweight='bold', y=1.02)

    plt.tight_layout()
    fig_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    Saved figure: {fig_path}")

    # ------------------------------------------------------------------
    # 6. Save outputs
    # ------------------------------------------------------------------
    print("\n[6] Saving outputs...")

    np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), phantom)
    np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), best_recon)

    metrics = {
        "task": "torch_radon_ct",
        "inverse_problem": "CT reconstruction via Radon transform inversion",
        "image_size": IMAGE_SIZE,
        "num_angles": NUM_ANGLES,
        "noise_snr_db": float(actual_snr),
        "best_method": best_method,
        "best_psnr_db": float(best_metrics['psnr']),
        "best_ssim": float(best_metrics['ssim']),
        "best_rmse": float(best_metrics['rmse']),
        "all_methods": {
            method: {
                "psnr_db": float(v['psnr']),
                "ssim": float(v['ssim']),
                "rmse": float(v['rmse']),
                "time_seconds": float(v['time'])
            } for method, v in results.items()
        }
    }
    with open(os.path.join(RESULTS_DIR, "metrics.json"), 'w') as f:
        json.dump(metrics, f, indent=2)

    print("    Saved: ground_truth.npy, reconstruction.npy, metrics.json")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  Ground truth:   {IMAGE_SIZE}x{IMAGE_SIZE} Shepp-Logan phantom")
    print(f"  Forward model:  Radon transform, {NUM_ANGLES} angles, SNR={actual_snr:.1f} dB")
    print(f"  Methods compared:")
    for method, v in results.items():
        print(f"    {method:14s}  PSNR={v['psnr']:.2f} dB  SSIM={v['ssim']:.4f}  RMSE={v['rmse']:.4f}  t={v['time']:.1f}s")
    print(f"  Best method:    {best_method}")
    print(f"  Best PSNR:      {best_metrics['psnr']:.2f} dB")
    print(f"  Best SSIM:      {best_metrics['ssim']:.4f}")
    print("=" * 60)

    return best_metrics

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `estimate_operator_norm`, `reconstruct_fbp`, `reconstruct_sirt`

In [ ]:
def estimate_operator_norm(theta, output_size, n_iter=15):
    """Estimate ||A^T A|| via power iteration for step size selection."""
    x = np.random.RandomState(0).randn(output_size, output_size)
    norm_est = 1.0
    for _ in range(n_iter):
        Ax = radon(x, theta=theta, circle=True)
        AtAx = iradon(Ax, theta=theta, output_size=output_size, filter_name=None, circle=True)
        norm_est = np.sqrt(np.sum(AtAx**2) / (np.sum(x**2) + 1e-10))
        x = AtAx / (np.linalg.norm(AtAx) + 1e-10) * np.linalg.norm(x)
    return norm_est

def reconstruct_fbp(sinogram, theta, output_size, filter_name='ramp'):
    """Filtered Back Projection reconstruction."""
    return iradon(sinogram, theta=theta, output_size=output_size,
                  filter_name=filter_name, circle=True)

def reconstruct_sirt(sinogram, theta, output_size, n_iter=200, init_fbp=False):
    """
    SIRT (Simultaneous Iterative Reconstruction Technique).
    
    Landweber iteration: x_{k+1} = x_k + step * A^T(y - A*x_k)
    with non-negativity constraint.
    """
    norm_est = estimate_operator_norm(theta, output_size)
    step = 0.9 / (norm_est + 1e-6)
    circle_mask = get_circle_mask(output_size)

    if init_fbp:
        x = iradon(sinogram, theta=theta, output_size=output_size,
                   filter_name='ramp', circle=True)
        x = np.maximum(x, 0) * circle_mask
    else:
        x = np.zeros((output_size, output_size))

    for i in range(n_iter):
        Ax = radon(x, theta=theta, circle=True)
        residual = sinogram - Ax
        gradient = iradon(residual, theta=theta, output_size=output_size,
                         filter_name=None, circle=True)
        x = x + step * gradient
        x = np.maximum(x, 0) * circle_mask

    return x

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`

In [ ]:
def compute_metrics(ground_truth, reconstruction):
    """Compute PSNR, SSIM, and RMSE between ground truth and reconstruction."""
    gt = ground_truth.copy()
    rec = reconstruction.copy()
    rec = np.clip(rec, 0, None)

    gt_max = gt.max()
    if gt_max > 0:
        gt_norm = gt / gt_max
        rec_norm = rec / gt_max
        rec_norm = np.clip(rec_norm, 0, 1)
    else:
        gt_norm = gt
        rec_norm = rec

    psnr = peak_signal_noise_ratio(gt_norm, rec_norm, data_range=1.0)
    ssim = structural_similarity(gt_norm, rec_norm, data_range=1.0)
    rmse = np.sqrt(mean_squared_error(gt_norm, rec_norm))
    return psnr, ssim, rmse

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **torch_radon_ct**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=30.01 dB, SSIM=0.8513)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: TorchRadon: Fast Differentiable Routines for Computed Tomography
- Repository: https://github.com/matteo-ronchetti/torch-radon
